# Machine Learning End-to-End Pipeline Examination
## Section A & Section B Implementation

**Dataset:** `messy_housing_dataset_large.csv`  
**Task:** Predictive Modeling of House Prices (Regression)  

---
### Table of Contents
1. **Section A: Dataset Cleaning & Feature Scaling (25 Marks)**
   - 1. Data Cleaning & Preprocessing (12 Marks)
   - 2. Categorical Encoding & Feature Scaling (13 Marks)
2. **Section B: Model Training, Tuning & Evaluation (35 Marks)**
   - 3. Data Splitting & Model Training (15 Marks)
   - 4. Model Evaluation & Persistence (20 Marks)
3. **Artifact Persistence & Deployment Verification**


## Section A: Dataset Cleaning & Feature Scaling (25 Marks)

### 1. Data Cleaning & Preprocessing (12 Marks)


In [1]:
import pandas as pd
import numpy as np

# Load dataset
df_raw = pd.read_csv('messy_housing_dataset_large.csv')

# Inspect raw dataset sample and missing values
print("Missing values count per column:")
print(df_raw.isna().sum())
df_raw.head()


Missing values count per column:
Customer_ID           0
Age                 182
Annual_Income       227
Years_Experience      0
Education_Level     121
City                 59
House_Price         152
dtype: int64


,Customer_ID,Age,Annual_Income,Years_Experience,Education_Level,City,House_Price
0,CUST-1000,56.0,45743.084346,38.161081,Bachelor,New York,396383.101328
1,CUST-1001,69.0,62128.604554,45.969812,high school,New York,523902.618142
2,CUST-1002,46.0,54900.455652,23.378285,Bachelor,Los Angeles,412974.803754
3,CUST-1003,NaN,90079.859772,8.484844,PhD,UNKNOWN,441150.551064
4,CUST-1004,60.0,6980.603832,38.531960,NaN,Chicago,187869.700803


In [2]:
# Create working copy
df = df_raw.copy()

# Step 1: Clean invalid / sentinel values
df['Age'] = df['Age'].apply(lambda x: np.nan if (pd.isna(x) or x < 0) else x)
df['Annual_Income'] = df['Annual_Income'].apply(lambda x: np.nan if (pd.isna(x) or x < 0) else x)
df['Years_Experience'] = df['Years_Experience'].apply(lambda x: np.nan if (pd.isna(x) or x == -999.0 or x < 0) else x)

# Standardize categorical string formats & replace sentinels with NaN
df['Education_Level'] = df['Education_Level'].replace(['UNKNOWN', '?', 'N/A', 'nan', 'None'], np.nan)
df['Education_Level'] = df['Education_Level'].apply(lambda x: x.strip().title() if isinstance(x, str) else x)
df['Education_Level'] = df['Education_Level'].replace(['Unknown', '?', 'N/A', 'Nan', 'None'], np.nan)

df['City'] = df['City'].replace(['UNKNOWN', '?', 'N/A', 'nan', 'None'], np.nan)
df['City'] = df['City'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df['City'] = df['City'].replace(['UNKNOWN', '?', 'N/A', 'nan', 'None', 'Nan'], np.nan)

# Drop rows missing target variable
df = df.dropna(subset=['House_Price']).reset_index(drop=True)

# Step 2: Median Imputation for Numerical Features
num_cols = ['Age', 'Annual_Income', 'Years_Experience']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Step 3: Mode Imputation for Categorical Features
cat_cols = ['Education_Level', 'City']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Inspect dataset after imputation
df.head()


,Customer_ID,Age,Annual_Income,Years_Experience,Education_Level,City,House_Price
0,CUST-1000,56.0,45743.084346,38.161081,Bachelor,New York,396383.101328
1,CUST-1001,69.0,62128.604554,45.969812,High School,New York,523902.618142
2,CUST-1002,46.0,54900.455652,23.378285,Bachelor,Los Angeles,412974.803754
3,CUST-1003,47.0,90079.859772,8.484844,Phd,New York,441150.551064
4,CUST-1004,60.0,6980.603832,38.531960,Bachelor,Chicago,187869.700803


In [3]:
# Outlier Detection and Removal via Interquartile Range (IQR) Method
num_all = ['Age', 'Annual_Income', 'Years_Experience', 'House_Price']
outlier_mask = pd.Series(True, index=df.index)

for col in num_all:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outlier_mask = outlier_mask & ((df[col] >= lower_bound) & (df[col] <= upper_bound))

# Filter dataframe
df_clean = df[outlier_mask].reset_index(drop=True)

print(f"Shape before IQR outlier removal: {df.shape}")
print(f"Shape after IQR outlier removal:  {df_clean.shape}")
df_clean.head()


Shape before IQR outlier removal: (1363, 7)
Shape after IQR outlier removal:  (1305, 7)


,Customer_ID,Age,Annual_Income,Years_Experience,Education_Level,City,House_Price
0,CUST-1000,56.0,45743.084346,38.161081,Bachelor,New York,396383.101328
1,CUST-1001,69.0,62128.604554,45.969812,High School,New York,523902.618142
2,CUST-1002,46.0,54900.455652,23.378285,Bachelor,Los Angeles,412974.803754
3,CUST-1003,47.0,90079.859772,8.484844,Phd,New York,441150.551064
4,CUST-1005,25.0,81830.137038,4.932717,High School,Houston,405855.885154


### 2. Categorical Encoding & Feature Scaling (13 Marks)

#### Theoretical Comparison: One-Hot Encoding vs. Label Encoding

| Aspect | **One-Hot Encoding** | **Label / Ordinal Encoding** |
|---|---|---|
| **Mechanism** | Creates binary columns ($0$ or $1$) for each unique categorical value. | Assigns an integer value ($0, 1, 2, \dots$) to each category. |
| **Ordinal Assumption** | Preserves no ordering; treats all categories as mutually independent. | Imposes an implicit numerical ordering ($0 < 1 < 2$). |
| **When to Use** | Use for **nominal** variables without ordering (e.g., `City` = New York, Los Angeles, Chicago). | Use for **ordinal** variables with clear rank (e.g., Education = High School < Bachelor < Master < PhD) or tree-based algorithms. |
| **Pros & Cons** | Prevents false ordinal distance assumptions; can increase dimensional complexity. | Keeps feature count compact; linear models may misinterpret arbitrary integer distances. |

---


In [4]:
# Separate Features (X) and Target (y)
X = df_clean.drop(columns=['Customer_ID', 'House_Price'])
y = df_clean['House_Price']

# Perform One-Hot Encoding on Categorical Features
X_encoded = pd.get_dummies(X, columns=['Education_Level', 'City'], drop_first=True, dtype=float)

# Inspect encoded features
X_encoded.head()


,Age,Annual_Income,Years_Experience,Education_Level_High School,Education_Level_Master,Education_Level_Phd,City_Houston,City_Los Angeles,City_New York,City_Phoenix
0,56.0,45743.084346,38.161081,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,69.0,62128.604554,45.969812,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,46.0,54900.455652,23.378285,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,47.0,90079.859772,8.484844,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,25.0,81830.137038,4.932717,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [5]:
from sklearn.preprocessing import StandardScaler
import joblib

# Identify numerical columns for scaling
num_features = ['Age', 'Annual_Income', 'Years_Experience']

# Instantiate and fit StandardScaler
scaler = StandardScaler()
scaler.fit(X_encoded[num_features])

# Transform numerical features
X_scaled = X_encoded.copy()
X_scaled[num_features] = scaler.transform(X_encoded[num_features])

# Save fitted scaler object to disk
joblib.dump(scaler, 'scaler.joblib')

# Inspect scaled features sample
X_scaled.head()


,Age,Annual_Income,Years_Experience,Education_Level_High School,Education_Level_Master,Education_Level_Phd,City_Houston,City_Los Angeles,City_New York,City_Phoenix
0,0.617840,-0.919935,0.787228,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.463204,0.060495,1.270186,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.032439,-0.372002,-0.127063,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.032589,1.732962,-1.048198,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,-1.398026,1.239338,-1.267892,1.0,0.0,0.0,1.0,0.0,0.0,0.0


#### Why Scaling Parameters Must Be Saved for Deployment
In machine learning deployment, incoming inference data points are unscaled. To make accurate predictions:
1. **Prevent Data Leakage:** The scaler must be fitted **only** on training data. Re-fitting the scaler on test or production data leads to target leakage and inconsistent feature representations.
2. **Consistent Transformation:** The deployment model expects features to be normalized using the exact mean ($\mu_{train}$) and standard deviation ($\sigma_{train}$) computed during training:
   $$z = \frac{x - \mu_{train}}{\sigma_{train}}$$
If new inputs were scaled with their own sample mean or left unscaled, feature magnitudes would mismatch model weights, causing degraded prediction accuracy.


## Section B: Model Training, Tuning & Evaluation (35 Marks)

### 3. Data Splitting & Model Training (15 Marks)


In [6]:
from sklearn.model_selection import train_test_split

# 80/20 Train-Test Split with fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Apply StandardScaler fitted on training set to prevent data leakage
scaler_train = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_features] = scaler_train.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler_train.transform(X_test[num_features])

# Save production scaler fitted on training set
joblib.dump(scaler_train, 'scaler.joblib')

print(f"Training Set Shape: {X_train_scaled.shape}")
print(f"Testing Set Shape:  {X_test_scaled.shape}")


Training Set Shape: (1044, 10)
Testing Set Shape:  (261, 10)


In [7]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

# Define Ridge Regression model and hyperparameter grid
ridge_model = Ridge(random_state=42)
param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 25.0, 50.0, 100.0, 250.0, 500.0]}

# Perform 5-Fold Cross-Validation GridSearchCV
grid_search = GridSearchCV(
    estimator=ridge_model,
    param_grid=param_grid,
    cv=5,
    scoring='r2'
)

# Fit GridSearchCV on Training Data
grid_search.fit(X_train_scaled, y_train)

# Extract Best Estimator
best_ridge_model = grid_search.best_estimator_

print("Best Hyperparameters:", grid_search.best_params_)
print(f"Best 5-Fold CV R2 Score: {grid_search.best_score_:.4f}")


Best Hyperparameters: {'alpha': 1.0}
Best 5-Fold CV R2 Score: 0.8607


### 4. Model Evaluation & Persistence (20 Marks)


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict on Test Set
y_pred_test = best_ridge_model.predict(X_test_scaled)

# Calculate Key Regression Evaluation Metrics
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

metrics_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2 Score'],
    'Value': [f"${mae_test:,.2f}", f"${rmse_test:,.2f}", f"{r2_test:.4f}"]
})
metrics_df


,Metric,Value
0,MAE,"$29,641.77"
1,RMSE,"$45,031.94"
2,R2 Score,0.8372


In [9]:
import pickle

# Export final trained model to disk as .pkl file
joblib.dump(best_ridge_model, 'model.pkl')
with open('model.pkl', 'wb') as f:
    pickle.dump(best_ridge_model, f)


### 5. Deployment Load Verification Test


In [10]:
# Load saved scaler and model from disk
loaded_scaler = joblib.load('scaler.joblib')
loaded_model = joblib.load('model.pkl')

# Test inference on sample raw input
raw_sample = pd.DataFrame([{
    'Age': 45.0,
    'Annual_Income': 75000.0,
    'Years_Experience': 20.0,
    'Education_Level': 'Master',
    'City': 'New York'
}])

sample_encoded = pd.get_dummies(raw_sample, columns=['Education_Level', 'City'], drop_first=True, dtype=float)
sample_aligned = sample_encoded.reindex(columns=X_train_scaled.columns, fill_value=0.0)
sample_aligned[num_features] = loaded_scaler.transform(sample_aligned[num_features])

predicted_price = loaded_model.predict(sample_aligned)[0]
print("Input Sample Payload:")
display(raw_sample)
print(f"\nPredicted House Price: ${predicted_price:,.2f}")


Input Sample Payload:


,Age,Annual_Income,Years_Experience,Education_Level,City
0,45.0,75000.0,20.0,Master,New York



Predicted House Price: $461,154.16
